# Section 3: Robustness, Where You Go Beyond (28 pts)

ECON 5200 Midterm Project · Replication Study

**AI is allowed in this notebook**, as a documented co-pilot under P.R.I.M.E. File every exchange, in full, in [`ai-appendix.md`](../ai-appendix.md) under the exercise it belongs to. If you cannot explain a line, you cannot submit it.

At least four exercises the original did not run. Choose from modern estimators, a specification curve, sensitivity to unobserved confounding, inference (clustering at the level of treatment assignment, a wild cluster bootstrap if clusters are few), bandwidth or instrument strength, and placebo tests, or justify an alternative. For each one: what you did, what you found, and what it implies about the original claim.

## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

ROOT = Path.cwd() if (Path.cwd() / "download_data.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

(ROOT / "output").mkdir(exist_ok=True)

from download_data import MY_PAPER, PAPERS, fetch

## Data

In [ ]:
paths = fetch(MY_PAPER)
card = pd.read_stata(paths[0])

print(PAPERS[MY_PAPER]["paper"])
print("Sample size:", len(card))

In [ ]:
CONTROLS = [
    "exper", "expersq", "black", "south", "smsa", "smsa66",
    "reg662", "reg663", "reg664", "reg665",
    "reg666", "reg667", "reg668", "reg669"
]

X = sm.add_constant(card[CONTROLS])

iv_fit = IV2SLS(
    card["lwage"],
    X,
    card["educ"],
    card["nearc4"]
).fit(cov_type="unadjusted", debiased=True)

print("Baseline return:", round(iv_fit.params["educ"], 4))
print("Baseline SE:", round(iv_fit.std_errors["educ"], 4))


def anderson_rubin(controls, data):
    grid = np.linspace(-2, 2, 4001)
    X = sm.add_constant(data[["nearc4"] + controls])
    keep = []

    for b in grid:
        outcome = data["lwage"] - b * data["educ"]
        result = sm.OLS(outcome, X).fit(cov_type="HC1")

        if result.pvalues["nearc4"] > 0.05:
            keep.append(b)

    if len(keep) == 0:
        return None

    edge = keep[0] == grid[0] or keep[-1] == grid[-1]

    return round(min(keep), 3), round(max(keep), 3), edge

## The four exercises

### Exercise 1: Heteroskedasticity-robust standard errors

**What I did.** I kept the same model but changed how I calculated the standard errors, since wages may be much more spread out for some groups of people than others. This checks whether allowing for those differences changes how confident we should be in the result.

In [ ]:
X = sm.add_constant(card[CONTROLS])

robust_fit = IV2SLS(
    card["lwage"],
    X,
    card["educ"],
    card["nearc4"]
).fit(cov_type="robust", debiased=True)

interval = robust_fit.conf_int().loc["educ"]

print("2SLS estimate:", round(robust_fit.params["educ"], 4))
print("Classical SE:", round(iv_fit.std_errors["educ"], 4))
print("Robust SE:", round(robust_fit.std_errors["educ"], 4))
print("Robust interval:", interval.round(4).tolist())

print("Return in log points:",
      round(robust_fit.params["educ"] * 100, 2))
print("Interval in log points:",
      (interval * 100).round(2).tolist())

**What I found.** The estimated return stays at 13.15 log points per extra year of schooling, while the standard error barely changes, going from 5.50 to 5.41 log points. The new 95% interval is [2.53, 23.77] log points, almost identical to the original interval of [2.4, 23.9].

**What it implies about the original claim.** - The positive result holds up when I allow wage errors to vary across people, since the standard error barely changes and the interval still stays above zero. This gives me more confidence that the finding isn’t especially sensitive to the original standard-error calculation. However, it leaves the bigger questions about college proximity open, because changing the standard errors cannot tell us whether proximity predicts schooling strongly enough or whether it also affects wages through local jobs and connections. I would take this as reassurance about how the model handles unequal wage variation, while still relying on the other checks to judge the causal claim.

### Exercise 2: Childhood metropolitan-area subsamples

**What I did.** I split the sample into men who grew up inside and outside metropolitan areas, since living near a college could mean something different depending on the area’s other opportunities; then I checked the schooling estimate and instrument strength separately for each group.

In [ ]:
area_controls = CONTROLS.copy()
area_controls.remove("smsa66")

for group in [0, 1]:
    sample = card[card["smsa66"] == group]

    X = sm.add_constant(sample[area_controls])

    area_fit = IV2SLS(
        sample["lwage"],
        X,
        sample["educ"],
        sample["nearc4"]
    ).fit(cov_type="robust", debiased=True)

    first_X = sm.add_constant(
        sample[["nearc4"] + area_controls]
    )

    first_fit = sm.OLS(
        sample["educ"], first_X
    ).fit(cov_type="HC1")

    ar = anderson_rubin(area_controls, sample)

    print("Metropolitan area in 1966:", group)
    print("Sample size:", len(sample))
    print("2SLS:", round(area_fit.params["educ"], 4))
    print("Robust SE:", round(area_fit.std_errors["educ"], 4))
    print("Robust first-stage F:",
          round(first_fit.tvalues["nearc4"] ** 2, 1))
    print("Return in log points:",
          round(area_fit.params["educ"] * 100, 2))
    print("Anderson-Rubin result:", ar)
    print("AR format: lower, upper, touches grid boundary")
    print()

**What I found.** Outside metropolitan areas, the estimated return is 16.01 log points per extra year of schooling, with an Anderson-Rubin interval of [3.2, 44.5] and a robust first-stage F of 8.2. Inside metropolitan areas, the return is 10.66 log points, with an interval of [-6.3, 34.8] and an F of 7.8, compared with the full-sample estimate of 13.15 log points and interval of [2.9, 28.1].

**What it implies about the original claim.** Both groups have positive estimates, but the wider intervals show that we know much less about the size of the return once we split the sample. For men who grew up inside metropolitan areas, the interval includes zero, so that group’s results are consistent with both no effect and a fairly large positive payoff. The intervals also overlap enough that the higher estimate outside metropolitan areas could easily reflect sampling uncertainty, so I wouldn’t claim that location changes the return based on these numbers alone. Some of the lost precision is understandable because each regression has fewer observations, and the weaker first stages make the estimates harder to pin down. The full-sample evidence for a positive return therefore remains more convincing than the evidence within either group separately, although this exercise cannot establish which location offers a more credible exclusion restriction.

### Exercise 3: Sensitivity to a direct effect of college proximity

**What I did.** I wondered what would happen if living near a college helped someone earn more through something besides schooling, such as better connections to local employers, so I tried a few possible direct wage effects and recalculated how much of the wage advantage would be left for schooling to explain.

In [ ]:
X = sm.add_constant(card[["nearc4"] + CONTROLS])

schooling_fit = sm.OLS(card["educ"], X).fit()
wage_fit = sm.OLS(card["lwage"], X).fit()

schooling_effect = schooling_fit.params["nearc4"]
wage_effect = wage_fit.params["nearc4"]

for direct_effect in [0, 0.01, 0.02, 0.04]:
    adjusted_return = (
        wage_effect - direct_effect
    ) / schooling_effect

    print("Assumed direct effect in log points:",
          round(direct_effect * 100, 2))
    print("Adjusted schooling coefficient:",
          round(adjusted_return, 4))
    print("Adjusted return in log points:",
          round(adjusted_return * 100, 2))
    print()

print("Direct effect that brings the return to zero:")
print("Coefficient:", round(wage_effect, 4))
print("Log points:", round(wage_effect * 100, 2))

**What I found.** The baseline return is 13.15 log points per extra year of schooling, but it falls to 10.02 if proximity directly adds one log point to wages and 6.90 if it directly adds two log points. A direct effect of four log points brings the schooling estimate down to 0.65, and about 4.21 log points would bring it to zero. These are estimates under assumed scenarios, and I did not calculate confidence intervals for them.

**What it implies about the original claim.** Card’s estimated return changes noticeably if we allow some of the wage advantage from college proximity to come through other channels. A direct benefit of just one log point lowers the schooling return from 13.15 to 10.02 log points, while a two-log-point benefit lowers it to 6.90, so fairly modest departures from the assumption could change how large the payoff appears. The point estimate stays positive in those scenarios, but it gets close to zero when the assumed direct benefit reaches four log points. That makes the exclusion restriction central to interpreting the result, because we need a convincing reason to attribute the proximity-related wage advantage to schooling. These scenarios show how sensitive the point estimate is, though they provide no evidence about whether those direct benefits actually exist, and I haven’t calculated intervals showing how uncertainty changes under each scenario.

### Exercise 4: Father’s education as a placebo outcome

**What I did.** I checked whether men who grew up near a college also tended to have more educated fathers, after accounting for race and the available childhood-location controls. This helps show whether the two groups came from different family backgrounds before considering the sons’ schooling and wages.

In [ ]:
placebo_controls = [
    "black", "smsa66",
    "reg662", "reg663", "reg664", "reg665",
    "reg666", "reg667", "reg668", "reg669"
]

needed = ["fatheduc", "nearc4"] + placebo_controls
placebo_sample = card.dropna(subset=needed)

X = sm.add_constant(
    placebo_sample[["nearc4"] + placebo_controls]
)

placebo_fit = sm.OLS(
    placebo_sample["fatheduc"], X
).fit(cov_type="HC1")

interval = placebo_fit.conf_int().loc["nearc4"]

print("Sample size:", len(placebo_sample))
print("Rows dropped:", len(card) - len(placebo_sample))
print("Coefficient:", round(placebo_fit.params["nearc4"], 4))
print("Robust SE:", round(placebo_fit.bse["nearc4"], 4))
print("P-value:", round(placebo_fit.pvalues["nearc4"], 4))
print("95% interval in years:", interval.round(4).tolist())

**What I found.** Fathers of men living near a college had an estimated 0.33 more years of schooling, with a standard error of 0.18 years and a 95% interval of about [-0.02, 0.68] years. The p-value is 0.0619, and missing information reduced the sample from 3,010 to 2,320 men.

**What it implies about the original claim.** The father’s education result gives only limited reassurance that men who grew up near and farther from colleges had comparable backgrounds, because the interval includes both zero and a potentially meaningful positive difference. The p-value of 0.0619 falls just above the usual 5% cutoff, but that cutoff alone gives little reason to dismiss the possibility of family differences. If more educated parents were more likely to live near colleges, their sons might also benefit from support, preparation or connections that affect later wages, which would make the identifying assumption harder to defend. This check does not tell us how much of the schooling estimate, if any, comes from those differences, especially since Card’s full analysis also considers family background. Losing 690 observations adds another limit, because the families with recorded father’s education may differ from those with missing information, so the result may not describe the full sample equally well.